# [Training a causal language model from scratch](https://huggingface.co/course/chapter7/6?fw=pt)
*This is part 2. Part 1 is **[here](xxx)***.

In [7]:
# reestablish required objects from part 1
## define tokenizer
from transformers import AutoTokenizer
from datasets import load_dataset, DatasetDict
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")
context_length = 128
## load and preprocess dataset
ds_train = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
ds_valid = load_dataset("huggingface-course/codeparrot-ds-valid", split="validation")
raw_datasets = DatasetDict(
    {
        "train": ds_train.shuffle().select(range(5000)), # originally .select(range(50000))
        "valid": ds_valid.shuffle().select(range(500))    # originally .select(range(500))
    }
)
def tokenize(element):
    outputs = tokenizer(
        element["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}
#
tokenized_datasets = raw_datasets.map(
    tokenize,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)
#
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig
config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id
)
model = GPT2LMHeadModel(config)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Training with 🤗 Accelerate
We've seen how to train a model with the `Trainer`, which can allow for some customization. However, sometimes we want full control over the training loop, or we want to make some exotic changes. In this case 🤗 Accelerate is a great choice, and in this section we'll go through the steps to use it to train our model. To make things more interesting, we’ll also add a twist to the training loop.

In [8]:
from IPython.display import HTML
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/Hm8_PgVTFuc" allowfullscreen></iframe>')

Since we are mainly interested in sensible autocompletion for the the data science libraries, it makes sense to give more weight to training samples that make more use of these libraries. We can easily identify these examples through the use of keywords such as `plt`, `pd`, `sk`, `fit`, and `predict`, which are the most frequent import names for `matplotlib.pyplot`, `pandas`, and `sklearn` as well as the fit/predict pattern of the latter. If these are each represented as a single token, we can easily check if they occur in the input sequence. Tokens can have a whitespace prefix, so we’ll also check for those versions in the tokenizer vocabulary. To verify that it works, we’ll add one test token which should be split into multiple tokens:

In [9]:
keytoken_ids = []
for keyword in ["plt", "pd", "sk", "fit", "predict", "accelerate"]:
    ids = tokenizer([keyword]).input_ids[0]
    if len(ids) == 1:
        keytoken_ids.append(ids[0])
    else:
        print(f"Keyword has number of tokens ≠ 1:\n{keyword}")

keytoken_ids

Keyword has number of tokens ≠ 1:
accelerate


[8436, 4289, 1201, 2770, 5431]

Great, that seems to work nicely! We can now write a custom loss function that takes the input sequence, the logits, and the key tokens we just selected as inputs. First, we need to align the logits and inputs: the input sequence shifted by one to the right forms the labels, since the next token is the label for the current token. We can achieve this by starting the labels from the second token of the input sequence, since the model does not make a prediction for the first token anyway. Then we cut off the last logit, as we don't have a label for the token that follows the full input sequence. With that we can compute the loss per sample and count the occurrences of all keywords in each sample. Finally, we calculate the weighted average over all samples using the occurrences as weights. Since we don't want to throw away all the samples that have no keywords, we add `1` to the weights:

In [10]:
import torch
from torch.nn import CrossEntropyLoss

def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    # Shift so that tokens < n predict n
    shift_labels = inputs[..., 1:].contiguous()
    shift_logits = logits[..., :-1, :].contiguous()
    # Calculate per-token loss
    loss_fct = CrossEntropyLoss(reduce=False)
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    # Resize and average loss per sample
    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
    # Calculate and scale weighting
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
        axis=[0, 2]
    )
    weights = alpha * (1.0 + weights)
    # Calculate weighted average
    weighted_loss = (loss_per_sample * weights).mean()
    return weighted_loss

keytoken_weighted_loss

<function __main__.keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0)>

Before we can start training with this awesome new loss function, we need to prepare a few things:
- We need dataloaders to load the data in batches.
- We need to set up weight decay parameters.
- From time to time we want to evaluate, so it makes sense to wrap the evaluation code in a function.

Let's start with the dataloaders. We only need to set the dataset's format to `"torch"`, and then we can pass it to a PyTorch `DataLoader` with the appropriate batch size:

In [11]:
from torch.utils.data.dataloader import DataLoader
tokenized_datasets.set_format("torch") # tokenized_datasets["train"].select(range(5000))
train_dataloader= DataLoader(tokenized_datasets["train"], batch_size=8, shuffle=True)
eval_dataloader = DataLoader(tokenized_datasets["valid"], batch_size=8)
train_dataloader, eval_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x7b7c2469bad0>,
 <torch.utils.data.dataloader.DataLoader at 0x7b7c2469a540>)

Next, we group the parameters so that the `optimizer` knows which ones will get an additional weight decay. Usually, all `bias` and `LayerNorm` weights terms are exempt from this; here's how we can do this:

In [12]:
def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [
        {"params": params_with_wd, "weight_decay": 0.1},
        {"params": params_without_wd, "weight_decay": 0.0},
    ]

print(f"{str(get_grouped_params(model))[:2000]} ...")

[{'params': [Parameter containing:
tensor([[ 0.0077, -0.0046, -0.0058,  ...,  0.0031, -0.0076,  0.0094],
        [ 0.0195,  0.0200, -0.0029,  ...,  0.0358,  0.0129,  0.0094],
        [-0.0116,  0.0138,  0.0026,  ...,  0.0394, -0.0201, -0.0010],
        ...,
        [-0.0139,  0.0138, -0.0210,  ...,  0.0182,  0.0039,  0.0110],
        [ 0.0128, -0.0028, -0.0310,  ..., -0.0062,  0.0163, -0.0538],
        [-0.0058,  0.0211, -0.0204,  ..., -0.0393,  0.0035,  0.0094]],
       requires_grad=True), Parameter containing:
tensor([[ 0.0106,  0.0178,  0.0197,  ...,  0.0041,  0.0219,  0.0138],
        [ 0.0090,  0.0057,  0.0305,  ...,  0.0094, -0.0018,  0.0380],
        [-0.0387, -0.0087,  0.0027,  ..., -0.0171, -0.0149, -0.0241],
        ...,
        [-0.0080, -0.0044,  0.0014,  ...,  0.0300,  0.0001,  0.0081],
        [ 0.0242, -0.0261, -0.0073,  ...,  0.0170,  0.0073,  0.0063],
        [ 0.0435,  0.0146, -0.0013,  ..., -0.0044, -0.0164,  0.0124]],
       requires_grad=True), Parameter containin

Since we want to evaluate the model regularly on the validation set during training, let's write a function for that as well. It just runs through the evaluation dataloader and gathers all the losses across processes:

In [13]:
def evaluate(model): # default version: evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            # https://stackoverflow.com/questions/63302534/how-to-write-torch-devicecuda-if-torch-cuda-is-available-else-cpu-as-a-f
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            outputs = model(batch["input_ids"].to(device), labels=batch["input_ids"].to(device))
        loss = outputs.loss
        losses.append(accelerator.gather(outputs.loss))
    # convert losses (=list) to tensor: https://discuss.pytorch.org/t/best-way-to-convert-a-list-to-a-tensor/59949/8
    losses = torch.FloatTensor(losses)
    loss = torch.mean(losses)
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        print("OverflowError!")
        perplexity = float("inf")
    return loss.item(), perplexity.item()

evaluate

<function __main__.evaluate(model)>

With the `evaluate()` function we can report loss and [perplexity](https://huggingface.co/course/chapter7/3) at regular intervals. Next, we redefine our model to make sure we train from scratch again:

In [14]:
model = GPT2LMHeadModel(config)
type(model)

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

We can then define our optimizer, using the function from before to split the parameters for weight decay:

In [15]:
from torch.optim import AdamW
# AdamW(params, lr=0.001
optimizer = AdamW(params=get_grouped_params(model), lr=5e-4)
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0.0
)

Now let's prepare the model, optimizer, and dataloaders so we can start training:

In [16]:
from accelerate import Accelerator
accelerator = Accelerator() # removed fp16=True argument
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)
optimizer, train_dataloader, eval_dataloader

(AcceleratedOptimizer (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0005
     maximize: False
     weight_decay: 0.1
 
 Parameter Group 1
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0005
     maximize: False
     weight_decay: 0.0
 ),
 <torch.utils.data.dataloader.DataLoader at 0x7b7c24614380>)

> 🚨 <font color="darkgreen">If you're training on a TPU, you'll need to move all the code starting at the cell above into a dedicated training function. See [Chapter 3](https://huggingface.co/course/chapter3) for more details.</font>

Now that we have sent our `train_dataloader` to `accelerator.prepare()`, we can use its length to compute the number of training steps. Remember that we should always do this after preparing the dataloader, as that method will change its length. We use a classic linear schedule from the learning rate to 0:

In [17]:
from transformers import get_scheduler
num_train_epochs = 2
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=1_000,
    num_training_steps=num_training_steps,
)
lr_scheduler

Lastly, to push our model to the Hub, we will need to create a `Repository` object in a working folder. First log in to the Hugging Face Hub, if you aren't logged in already. We'll determine the repository name from the model ID we want to give our model (feel free to replace the `repo_name` with your own choice; it just needs to contain your username, which is what the function `get_full_repo_name()` does):

In [18]:
from huggingface_hub import Repository, get_full_repo_name, create_repo
model_name = "codeparrot-model-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'mdroth/codeparrot-model-accelerate'

Then we can clone that repository in a local folder. If it already exists, this local folder should be an existing clone of the repository we are working with:

In [19]:
output_dir = "sections/section_7/logs/codeparrot-ds-accelerate"
# try to get repo
try:
    repo = Repository(output_dir, clone_from=repo_name)
    repo_message = f"The '{repo_name}' repo has already been created."
# otherwise, create repo
except:
    repo = create_repo(repo_name)
    repo_message = f"The '{repo_name}' repo has just been created."
print(repo_message)

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)


The 'mdroth/codeparrot-model-accelerate' repo has just been created.


We can now upload anything we save in `output_dir` by calling the `repo.push_to_hub()` method. This will help us upload the intermediate models at the end of each epoch.

Before we train, let's run a quick test to see if the evaluation function works properly:

In [20]:
evaluate(model=model)

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


(10.934571266174316, 56082.0625)

Those are very high values for loss and perplexity, but that's not surprising as we haven't trained the model yet. With that, we have everything prepared to write the core part of the training script: the training loop. In the training loop we iterate over the dataloader and pass the batches to the model. With the logits, we can then evaluate our custom loss function. We scale the loss by the number of gradient accumulation steps so as not to create larger losses when aggregating more steps. Before we optimize, we also clip the gradients for better convergence. Finally, every few steps we evaluate the model on the evaluation set with our new `evaluate()` function:

In [ ]:
from tqdm.notebook import tqdm
eval_steps = 5_000
model.train()
samples_per_step = 1
completed_steps = 0
gradient_accumulation_steps = 8
for epoch in range(num_train_epochs):
    for step, batch in tqdm(enumerate(train_dataloader, start=1), total=num_training_steps):
        logits = model(batch["input_ids"]).logits
        loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
        if step % 45_000 == 0:
            accelerator.print(
                {
                    "samples": step * samples_per_step,
                    "steps": completed_steps,
                    "loss/train": loss.item() * gradient_accumulation_steps,
                }
            )
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        if step % gradient_accumulation_steps == 0:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            completed_steps += 1
        if (step % (eval_steps * gradient_accumulation_steps)) == 0:
            eval_loss, perplexity = evaluate(model=model) # evaluate() contains model.eval()
            accelerator.print({"loss/eval": eval_loss, "perplexity": perplexity})
            model.train()
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
            if accelerator.is_main_process:
                tokenizer.save_pretrained(output_dir)
                repo.push_to_hub(commit_message=f"Training in progress step {step}", blocking=False)

  0%|          | 0/33356 [00:00<?, ?it/s]

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


  0%|          | 0/33356 [00:00<?, ?it/s]

And that's it — you now have your own custom training loop for causal language models such as GPT-2 that you can further customize to your needs.

> ✏️ Try it out! <font color="darkgreen">Either create your own custom loss function tailored to your use case, or add another custom step into the training loop.</font>

In [ ]:
###########################################################################################
# trying it out 3                                                                         #
# use another custom step: counting flops                                                 #
###########################################################################################
# - Use a dataset subset and a custom loss function: cross entropy!                       #
# - Also check the article on cost functions for causal language models at ...            #
# - ... "Huggingface-course/sections/section_7/CrossEntropy_NLL_Perplexity_Tutorial.pdf"! #
###########################################################################################
eval_steps = 5_000
samples_per_step = 1
completed_steps = 0
gradient_accumulation_steps = 8
accelerator = Accelerator()
for epoch in range(3):
    model.train()
    for step, batch in tqdm(enumerate(train_dataloader, start=1), total=num_training_steps):
        logits = model(batch["input_ids"]).logits
        loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
        if step % 45_000 == 0:
            accelerator.print(
                {
                    "samples": step * samples_per_step,
                    "steps": completed_steps,
                    "loss/train": loss.item() * gradient_accumulation_steps,
                }
            )
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        if step % gradient_accumulation_steps == 0:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            completed_steps += 1
        if (step % (eval_steps * gradient_accumulation_steps)) == 0:
            model.eval()
            eval_loss, perplexity = evaluate(model=model) # evaluate() contains model.eval()
            accelerator.print({"loss/eval": eval_loss, "perplexity": perplexity})
            model.train()
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
            if accelerator.is_main_process:
                tokenizer.save_pretrained(output_dir)
                repo.push_to_hub(commit_message=f"Training in progress step {step}", blocking=False)

> ✏️ Try it out! <font color="darkgreen">When running long training experiments it's a good idea to log important metrics using tools such as TensorBoard or Weights & Biases. Add proper logging to the training loop so you can always check how the training is going.</font>

In [ ]:
############################################
# trying it out 4: Add logging with wandb! #
############################################


$\checkmark$